In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df_function = pd.read_excel("../data/01_FunctionAnalysis_data.xlsx")
df_force = pd.read_excel("../data/02_ForceAnalysis_data.xlsx")
df_torque = pd.read_excel("../data/03_TorqueAnalysis_Data.xlsx")
df_torque.rename(columns={"Paciente": "ID"}, inplace=True)

In [3]:
df_total_temp = df_function.merge(df_force, on=["ID", "Gender", "BMI_Value", "AgeGroup", "Clinical Treatment"])

df = df_total_temp.merge(df_torque, on=["ID", "Gender", "BMI_Value", "AgeGroup", "Clinical Treatment"])

df.columns, df.shape

(Index(['ID', 'Gender', 'BMI_Value', 'AgeGroup', 'Clinical Treatment',
        'tSTRIDE', 'tSTANCE', 'tSWING', 'tDBLSTANCE', 'velMEAN', 'fGRVE',
        'fGRAP', 'fGRML', 'fAML', 'fKML', 'fKAP', 'fKCP', 'fHPML', 'fHPAP',
        'fHPCP', 'tGRVE', 'tAFE', 'tKFE', 'tKAA', 'tKIE', 'tHPFE', 'tHPAA',
        'tHPIE'],
       dtype='str'),
 (81, 28))

In [5]:
pd.crosstab(
    df["Clinical Treatment"],
    df["Gender"]
)

Gender,Female,Male
Clinical Treatment,,
Conservative,25,21
TKR-planned,24,11


In [ ]:
groups = df.groupby(
    ["Clinical Treatment", "Gender"]
)["tKAA"]

groups.agg(["count", "mean", "std"])

count      mean       std
Clinical Treatment Gender                           
Conservative       Female     25  0.337480  0.103607
                   Male       21  0.372619  0.140493
TKR-planned        Female     24  0.455583  0.126578
                   Male       11  0.410455  0.140450

In [7]:
grand_mean = df["tKAA"].mean()

treatment_means = df.groupby("Clinical Treatment")["tKAA"].mean()
gender_means = df.groupby("Gender")["tKAA"].mean()

grand_mean, treatment_means, gender_means

(np.float64(0.3914938271604938),
 Clinical Treatment
 Conservative    0.353522
 TKR-planned     0.441400
 Name: tKAA, dtype: float64,
 Gender
 Female    0.395327
 Male      0.385625
 Name: tKAA, dtype: float64)

In [ ]:
SS_total = ((df["tKAA"] - grand_mean) ** 2).sum()

SS_total

np.float64(1.4011102469135803)

In [11]:
SS_error = 0

for (treatment, gender), group in df.groupby(
    ["Clinical Treatment", "Gender"]
):
    group_mean = group["tKAA"].mean()

    for value in group["tKAA"]:
        SS_error += (value - group_mean) ** 2

SS_error

np.float64(1.2181577529870131)

In [12]:
SS_model = SS_total - SS_error
SS_model

np.float64(0.1829524939265672)

In [13]:
df.groupby(["Clinical Treatment", "Gender"])["tKAA"].mean()

Clinical Treatment  Gender
Conservative        Female    0.337480
                    Male      0.372619
TKR-planned         Female    0.455583
                    Male      0.410455
Name: tKAA, dtype: float64

In [14]:
N = len(df)
a = df["Clinical Treatment"].nunique()
b = df["Gender"].nunique()

df_error = N - a * b
MS_error = SS_error / df_error

df_error, MS_error

(77, np.float64(0.015820230558272898))

In [15]:
cell_means = df.groupby(
    ["Clinical Treatment", "Gender"]
)["tKAA"].mean()

treatment_means = df.groupby(
    "Clinical Treatment"
)["tKAA"].mean()

gender_means = df.groupby(
    "Gender"
)["tKAA"].mean()

grand_mean = df["tKAA"].mean()

for (treatment, gender), cell_mean in cell_means.items():
    interaction_deviation = (
        cell_mean
        - treatment_means[treatment]
        - gender_means[gender]
        + grand_mean
    )

    print(
        treatment,
        gender,
        interaction_deviation
    )

Conservative Female -0.019874442582185847
Conservative Male 0.024966135649106658
TKR-planned Female 0.010350629881582307
TKR-planned Male -0.025076627384960803


In [16]:
interaction_contrast = (
    (0.455583 - 0.337480)
    - (0.410455 - 0.372619)
)

inverse_n_sum = (
    1/25
    + 1/21
    + 1/24
    + 1/11
)

interaction_contrast, inverse_n_sum

(0.08026699999999998, 0.2201948051948052)

In [17]:
F_interaction = (
    interaction_contrast ** 2
    / (MS_error * inverse_n_sum)
)

F_interaction

np.float64(1.849499360232399)

In [19]:
from scipy.stats import f

df_interaction = (a - 1) * (b - 1)

p_interaction = f.sf(
    F_interaction,
    df_interaction,
    df_error
)

p_interaction

np.float64(0.1778102986168822)

In [20]:
mean_conservative = (
    0.337480 + 0.372619
) / 2

mean_tkr = (
    0.455583 + 0.410455
) / 2

treatment_contrast = mean_tkr - mean_conservative

mean_conservative, mean_tkr, treatment_contrast

(0.3550495, 0.43301900000000004, 0.07796950000000002)

In [21]:
var_treatment = MS_error * (
    (0.5**2) / 25
    + (0.5**2) / 21
    + (0.5**2) / 24
    + (0.5**2) / 11
)

var_treatment

np.float64(0.0008708831464789513)

In [22]:
F_treatment = treatment_contrast**2 / var_treatment
F_treatment

np.float64(6.980549520138102)

In [23]:
df_treatment = a - 1

p_treatment = f.sf(
    F_treatment,
    df_treatment,
    df_error
)

p_treatment

np.float64(0.009976376429142836)

In [24]:
mean_female = (
    0.337480 + 0.455583
) / 2

mean_male = (
    0.372619 + 0.410455
) / 2

gender_contrast = mean_female - mean_male

mean_female, mean_male, gender_contrast

(0.39653150000000004, 0.391537, 0.004994500000000013)

In [25]:
var_gender = MS_error * (
    (0.5**2) / 25
    + (0.5**2) / 24
    + (0.5**2) / 21
    + (0.5**2) / 11
)

var_gender

np.float64(0.0008708831464789513)

In [26]:
F_gender = gender_contrast**2 / var_gender
F_gender

np.float64(0.02864337236385253)

In [27]:
df_gender = b - 1

p_gender = f.sf(
    F_gender,
    df_gender,
    df_error
)

p_gender

np.float64(0.8660489203865116)

In [28]:
SS_treatment = F_treatment * MS_error
SS_gender = F_gender * MS_error
SS_interaction = F_interaction * MS_error

SS_treatment, SS_gender, SS_interaction

(np.float64(0.11043390283202602),
 np.float64(0.00045314475476260923),
 np.float64(0.029259506296254772))

In [29]:
anova_table = pd.DataFrame({
    "SS": [
        SS_treatment,
        SS_gender,
        SS_interaction,
        SS_error
    ],
    "df": [
        1,
        1,
        1,
        df_error
    ],
    "MS": [
        SS_treatment,
        SS_gender,
        SS_interaction,
        MS_error
    ],
    "F": [
        F_treatment,
        F_gender,
        F_interaction,
        np.nan
    ],
    "p-value": [
        p_treatment,
        p_gender,
        p_interaction,
        np.nan
    ]
}, index=[
    "Clinical Treatment",
    "Gender",
    "Clinical Treatment × Gender",
    "Error"
])

anova_table

,SS,df,MS,F,p-value
Clinical Treatment,0.110434,1,0.110434,6.980550,0.009976
Gender,0.000453,1,0.000453,0.028643,0.866049
Clinical Treatment × Gender,0.029260,1,0.029260,1.849499,0.177810
Error,1.218158,77,0.015820,NaN,NaN


## To-do:

- Understand the code and start writing the document
- Verify all the code is correct 
- Make plots to support ideas and hypothesis